<a href="https://colab.research.google.com/github/nehansa2003/Weather-Forecasting-and-Analysis-Using-LSTM/blob/main/notebooks/6_training_baseline_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q xgboost lightgbm

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    r2_score,
    accuracy_score,
    roc_auc_score,
    mean_absolute_error,
    mean_squared_error,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor

import warnings
warnings.filterwarnings("ignore")

In [ ]:
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
file_path = "/content/drive/MyDrive/Capstone_Datasets/colombo_weather.parquet"

weather_df = pd.read_parquet(file_path)
weather_df["datetime"] = pd.to_datetime(
    weather_df["datetime"]
)

weather_df = weather_df.sort_values(
    ["district", "pointid", "datetime"]
).reset_index(drop=True)

print("Data sorted successfully.")

Data sorted successfully.


In [ ]:
train_df = weather_df[
    weather_df["year"] <= 2023
].copy()

val_df = weather_df[
    weather_df["year"] == 2024
].copy()

test_df = weather_df[
    weather_df["year"] == 2025
].copy()

In [ ]:
group_cols = [
    "district",
    "pointid",
    "hour"
]

train_df["rain_occurs"] = (
    train_df["rainfall_mm"] >= 0.1
).astype("int8")

group_sum = train_df.groupby(
    group_cols
)["rain_occurs"].transform("sum")

group_count = train_df.groupby(
    group_cols
)["rain_occurs"].transform("count")

train_df["historical_rain_probability_clean"] = np.where(
    group_count > 1,
    (group_sum - train_df["rain_occurs"]) /
    (group_count - 1),
    group_sum / group_count
)

In [ ]:
historical_probability = (
    train_df
    .groupby(group_cols)["rain_occurs"]
    .mean()
    .reset_index(name="historical_rain_probability_clean")
)

In [ ]:
val_df = val_df.drop(
    columns=["historical_rain_probability_clean"],
    errors="ignore"
)

test_df = test_df.drop(
    columns=["historical_rain_probability_clean"],
    errors="ignore"
)

In [ ]:
print(
    train_df[
        [
            "district",
            "pointid",
            "hour",
            "rain_occurs",
            "historical_rain_probability_clean"
        ]
    ].head()
)

  district  pointid  hour  rain_occurs  historical_rain_probability_clean
0  Colombo        1     0            1                           0.413699
1  Colombo        1     1            0                           0.435616
2  Colombo        1     2            1                           0.445890
3  Colombo        1     3            1                           0.460274
4  Colombo        1     4            0                           0.472603


Create lag features

In [ ]:
weather_variables = [
    "temperature_c",
    "humidity_pct",
    "pressure_hpa",
    "windspeed_kmh",
    "rainfall_mm",
    "windgust_kmh",
    "cloudcover_pct",
    "solarradiation_wm2",
    "dewpoint_c"
]

lags = [1, 3, 6, 12, 24]

In [ ]:
def create_lag_features(df):

    df = df.copy()

    df = df.sort_values(
        ["district", "pointid", "datetime"]
    )

    for col in weather_variables:

        if col in df.columns:

            for lag in lags:

                df[f"{col}_lag_{lag}"] = (
                    df
                    .groupby(
                        ["district", "pointid"]
                    )[col]
                    .shift(lag)
                )

    return df

In [ ]:
train_tree = create_lag_features(train_df)
val_tree = create_lag_features(val_df)
test_tree = create_lag_features(test_df)

Add rolling rainfall features

In [ ]:
def add_rainfall_rolling_features(df):

    df = df.copy()

    # Make sure data is in correct temporal order
    df = df.sort_values(
        ["district", "pointid", "datetime"]
    ).reset_index(drop=True)

    # Previous rainfall only
    # shift(1) prevents using the current hour's rainfall
    previous_rain = (
        df.groupby(
            ["district", "pointid"]
        )["rainfall_mm"]
        .shift(1)
    )

    # Rolling rainfall based only on previous hours
    df["rainfall_rolling_3h"] = (
        previous_rain
        .groupby([
            df["district"],
            df["pointid"]
        ])
        .transform(
            lambda x: x.rolling(3, min_periods=1).sum()
        )
    )

    df["rainfall_rolling_6h"] = (
        previous_rain
        .groupby([
            df["district"],
            df["pointid"]
        ])
        .transform(
            lambda x: x.rolling(6, min_periods=1).sum()
        )
    )

    df["rainfall_rolling_12h"] = (
        previous_rain
        .groupby([
            df["district"],
            df["pointid"]
        ])
        .transform(
            lambda x: x.rolling(12, min_periods=1).sum()
        )
    )

    df["rainfall_rolling_24h"] = (
        previous_rain
        .groupby([
            df["district"],
            df["pointid"]
        ])
        .transform(
            lambda x: x.rolling(24, min_periods=1).sum()
        )
    )

    return df

In [ ]:
train_tree = add_rainfall_rolling_features(train_tree)
val_tree = add_rainfall_rolling_features(val_tree)
test_tree = add_rainfall_rolling_features(test_tree)

In [ ]:
time_features = [
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos"
]

In [ ]:
base_features = [
    "temperature_c",
    "humidity_pct",
    "pressure_hpa",
    "windspeed_kmh",
    "windgust_kmh",
    "cloudcover_pct",
    "solarradiation_wm2",
    "dewpoint_c",

    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",

    "historical_rain_probability_clean",

    "rainfall_rolling_3h",
    "rainfall_rolling_6h",
    "rainfall_rolling_12h",
    "rainfall_rolling_24h"
]

In [ ]:
lag_feature_names = []

for col in weather_variables:

    for lag in lags:

        feature_name = f"{col}_lag_{lag}"

        if feature_name in train_tree.columns:

            lag_feature_names.append(
                feature_name
            )

In [ ]:
features_tree = (
    base_features +
    lag_feature_names
)

features_tree = [
    col for col in features_tree
    if col in train_tree.columns
]

print("Number of features:", len(features_tree))

print("\nFeatures:")
for feature in features_tree:
    print(feature)

Number of features: 62

Features:
temperature_c
humidity_pct
pressure_hpa
windspeed_kmh
windgust_kmh
cloudcover_pct
solarradiation_wm2
dewpoint_c
hour_sin
hour_cos
month_sin
month_cos
historical_rain_probability_clean
rainfall_rolling_3h
rainfall_rolling_6h
rainfall_rolling_12h
rainfall_rolling_24h
temperature_c_lag_1
temperature_c_lag_3
temperature_c_lag_6
temperature_c_lag_12
temperature_c_lag_24
humidity_pct_lag_1
humidity_pct_lag_3
humidity_pct_lag_6
humidity_pct_lag_12
humidity_pct_lag_24
pressure_hpa_lag_1
pressure_hpa_lag_3
pressure_hpa_lag_6
pressure_hpa_lag_12
pressure_hpa_lag_24
windspeed_kmh_lag_1
windspeed_kmh_lag_3
windspeed_kmh_lag_6
windspeed_kmh_lag_12
windspeed_kmh_lag_24
rainfall_mm_lag_1
rainfall_mm_lag_3
rainfall_mm_lag_6
rainfall_mm_lag_12
rainfall_mm_lag_24
windgust_kmh_lag_1
windgust_kmh_lag_3
windgust_kmh_lag_6
windgust_kmh_lag_12
windgust_kmh_lag_24
cloudcover_pct_lag_1
cloudcover_pct_lag_3
cloudcover_pct_lag_6
cloudcover_pct_lag_12
cloudcover_pct_lag_24
solarr

Remove rows created with missing lag values

In [ ]:
print("Train:", "historical_rain_probability_clean" in train_tree.columns)
print("Validation:", "historical_rain_probability_clean" in val_tree.columns)
print("Test:", "historical_rain_probability_clean" in test_tree.columns)

Train: True
Validation: False
Test: False


In [ ]:
# ============================================================
# ADD TRAINING-DERIVED HISTORICAL RAIN PROBABILITY
# ============================================================

prob_keys = [
    "district",
    "pointid",
    "hour"
]

prob_col = "historical_rain_probability_clean"

# Add probability to validation data
val_tree = val_tree.merge(
    historical_probability,
    on=prob_keys,
    how="left"
)

# Add probability to test data
test_tree = test_tree.merge(
    historical_probability,
    on=prob_keys,
    how="left"
)

print(
    "Validation probability column:",
    prob_col in val_tree.columns
)

print(
    "Test probability column:",
    prob_col in test_tree.columns
)

Validation probability column: True
Test probability column: True


In [ ]:
train_tree = train_tree.dropna(
    subset=features_tree
).reset_index(drop=True)

val_tree = val_tree.dropna(
    subset=features_tree
).reset_index(drop=True)

test_tree = test_tree.dropna(
    subset=features_tree
).reset_index(drop=True)

In [ ]:
print("Train:", train_tree.shape)
print("Validation:", val_tree.shape)
print("Test:", test_tree.shape)

Train: (385440, 85)
Validation: (96360, 84)
Test: (96096, 84)


In [ ]:


RAIN_THRESHOLD = 0.1

train_tree["rain_occurs"] = (
    train_tree["rainfall_mm"] >= RAIN_THRESHOLD
).astype("int8")

val_tree["rain_occurs"] = (
    val_tree["rainfall_mm"] >= RAIN_THRESHOLD
).astype("int8")

test_tree["rain_occurs"] = (
    test_tree["rainfall_mm"] >= RAIN_THRESHOLD
).astype("int8")

In [ ]:
X_train = train_tree[features_tree]
X_val = val_tree[features_tree]
X_test = test_tree[features_tree]

y_train_class = train_tree["rain_occurs"]
y_val_class = val_tree["rain_occurs"]
y_test_class = test_tree["rain_occurs"]

In [ ]:
y_train_rain = train_tree["rainfall_mm"]
y_val_rain = val_tree["rainfall_mm"]
y_test_rain = test_tree["rainfall_mm"]

In [ ]:
print(
    "Training rainfall statistics:"
)

print(
    y_train_rain.describe()
)

Training rainfall statistics:
count    385440.000000
mean          0.332582
std           0.971372
min           0.000000
25%           0.000000
50%           0.000000
75%           0.200000
max          58.700000
Name: rainfall_mm, dtype: float64


In [ ]:
print(
    "\nRain occurrence:"
)

print(
    y_train_class.value_counts()
)

print(
    y_train_class.value_counts(
        normalize=True
    ) * 100
)


Rain occurrence:
rain_occurs
0    199543
1    185897
Name: count, dtype: int64
rain_occurs
0    51.770185
1    48.229815
Name: proportion, dtype: float64


RANDOM FOREST

Rain / No-Rain classifier

In [ ]:
rf_classifier = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_classifier.fit(
    X_train,
    y_train_class
)

print("Random Forest classifier trained.")

Random Forest classifier trained.


Random Forest validation prediction

In [ ]:
rf_val_probability = (
    rf_classifier
    .predict_proba(X_val)[:, 1]
)

rf_val_prediction = (
    rf_val_probability >= 0.5
).astype(int)

In [ ]:
print(
    "Random Forest Validation"
)

print(
    "Precision:",
    precision_score(
        y_val_class,
        rf_val_prediction,
        zero_division=0
    )
)

print(
    "Recall:",
    recall_score(
        y_val_class,
        rf_val_prediction,
        zero_division=0
    )
)

print(
    "F1:",
    f1_score(
        y_val_class,
        rf_val_prediction,
        zero_division=0
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_val_class,
        rf_val_probability
    )
)

Random Forest Validation
Precision: 0.834485840124258
Recall: 0.8718461709458708
F1: 0.8527570012176031
ROC-AUC: 0.9300500402540235


Random Forest rainfall amount model

In [ ]:
rain_train_rf = train_tree[
    train_tree["rain_occurs"] == 1
]

X_train_rf_rain = rain_train_rf[
    features_tree
]

y_train_rf_rain = rain_train_rf[
    "rainfall_mm"
]

print(
    "Rainy training samples:",
    len(rain_train_rf)
)

Rainy training samples: 185897


In [ ]:
rf_regressor = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
    verbose=10
)

rf_regressor.fit(
    X_train_rf_rain,
    y_train_rf_rain
)

print("Random Forest rainfall regressor trained.")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.


building tree 1 of 300
building tree 2 of 300
building tree 3 of 300


[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:   19.6s


building tree 4 of 300
building tree 5 of 300
building tree 6 of 300


[Parallel(n_jobs=-1)]: Done   4 tasks      | elapsed:   31.5s


...

[Parallel(n_jobs=-1)]: Done 261 tasks      | elapsed: 30.1min


building tree 264 of 300
building tree 265 of 300
building tree 266 of 300
building tree 267 of 300
building tree 268 of 300
building tree 269 of 300
building tree 270 of 300
building tree 271 of 300
building tree 272 of 300
building tree 273 of 300
building tree 274 of 300
building tree 275 of 300
building tree 276 of 300
building tree 277 of 300
building tree 278 of 300
building tree 279 of 300
building tree 280 of 300
building tree 281 of 300
building tree 282 of 300
building tree 283 of 300
building tree 284 of 300
building tree 285 of 300
building tree 286 of 300


[Parallel(n_jobs=-1)]: Done 284 tasks      | elapsed: 32.7min


building tree 287 of 300
building tree 288 of 300
building tree 289 of 300
building tree 290 of 300
building tree 291 of 300
building tree 292 of 300
building tree 293 of 300
building tree 294 of 300
building tree 295 of 300
building tree 296 of 300
building tree 297 of 300
building tree 298 of 300
building tree 299 of 300
building tree 300 of 300
Random Forest rainfall regressor trained.


[Parallel(n_jobs=-1)]: Done 300 out of 300 | elapsed: 34.4min finished


Random Forest final rainfall prediction

In [ ]:
rf_test_probability = (
    rf_classifier
    .predict_proba(X_test)[:, 1]
)

rf_test_rain_prediction = (
    rf_test_probability >= 0.5
).astype(int)

In [ ]:
rf_test_amount = (
    rf_regressor
    .predict(X_test)
)

rf_test_amount = np.maximum(
    rf_test_amount,
    0
)

[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   1 tasks      | elapsed:    0.1s
[Parallel(n_jobs=2)]: Done   4 tasks      | elapsed:    0.1s
[Parallel(n_jobs=2)]: Done   9 tasks      | elapsed:    0.3s
[Parallel(n_jobs=2)]: Done  14 tasks      | elapsed:    0.4s
[Parallel(n_jobs=2)]: Done  21 tasks      | elapsed:    0.6s
[Parallel(n_jobs=2)]: Done  28 tasks      | elapsed:    0.8s
[Parallel(n_jobs=2)]: Done  37 tasks      | elapsed:    1.1s
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    1.4s
[Parallel(n_jobs=2)]: Done  57 tasks      | elapsed:    1.7s
[Parallel(n_jobs=2)]: Done  68 tasks      | elapsed:    2.0s
[Parallel(n_jobs=2)]: Done  81 tasks      | elapsed:    2.4s
[Parallel(n_jobs=2)]: Done  94 tasks      | elapsed:    2.6s
[Parallel(n_jobs=2)]: Done 109 tasks      | elapsed:    2.9s
[Parallel(n_jobs=2)]: Done 124 tasks      | elapsed:    3.1s
[Parallel(n_jobs=2)]: Done 141 tasks      | elapsed:    3.3s
[Para

In [ ]:
rf_final_prediction = np.where(
    rf_test_rain_prediction == 1,
    rf_test_amount,
    0
)

XGBOOST

Classifier

In [ ]:
xgb_classifier = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    verbose=1
)

xgb_classifier.fit(
    X_train,
    y_train_class
)

print("XGBoost classifier trained.")

XGBoost classifier trained.


In [ ]:
xgb_val_probability = (
    xgb_classifier
    .predict_proba(X_val)[:, 1]
)

xgb_val_prediction = (
    xgb_val_probability >= 0.5
).astype(int)

XGBoost rainfall regressor

In [ ]:
rain_train_xgb = train_tree[
    train_tree["rain_occurs"] == 1
]

X_train_xgb_rain = rain_train_xgb[
    features_tree
]

y_train_xgb_rain = rain_train_xgb[
    "rainfall_mm"
]

In [ ]:
xgb_regressor = XGBRegressor(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    eval_metric="rmse",
    random_state=42,
    n_jobs=-1,
    verbose=1
)

xgb_regressor.fit(
    X_train_xgb_rain,
    y_train_xgb_rain
)

print("XGBoost rainfall regressor trained.")

XGBoost rainfall regressor trained.


In [ ]:
xgb_test_probability = (
    xgb_classifier
    .predict_proba(X_test)[:, 1]
)

xgb_test_rain_prediction = (
    xgb_test_probability >= 0.5
).astype(int)

xgb_test_amount = (
    xgb_regressor
    .predict(X_test)
)

xgb_test_amount = np.maximum(
    xgb_test_amount,
    0
)

xgb_final_prediction = np.where(
    xgb_test_rain_prediction == 1,
    xgb_test_amount,
    0
)

LIGHTGBM

Classifier

In [ ]:
lgb_classifier = LGBMClassifier(
    n_estimators=300,
    max_depth=10,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgb_classifier.fit(
    X_train,
    y_train_class
)

print("LightGBM classifier trained.")

LightGBM classifier trained.


In [ ]:
lgb_val_probability = (
    lgb_classifier
    .predict_proba(X_val)[:, 1]
)

lgb_val_prediction = (
    lgb_val_probability >= 0.5
).astype(int)

LightGBM rainfall regressor

In [ ]:
rain_train_lgb = train_tree[
    train_tree["rain_occurs"] == 1
]

X_train_lgb_rain = rain_train_lgb[
    features_tree
]

y_train_lgb_rain = rain_train_lgb[
    "rainfall_mm"
]

In [ ]:
lgb_regressor = LGBMRegressor(
    n_estimators=300,
    max_depth=10,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="regression",
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgb_regressor.fit(
    X_train_lgb_rain,
    y_train_lgb_rain
)

print("LightGBM rainfall regressor trained.")

LightGBM rainfall regressor trained.


In [ ]:
lgb_test_probability = (
    lgb_classifier
    .predict_proba(X_test)[:, 1]
)

lgb_test_rain_prediction = (
    lgb_test_probability >= 0.5
).astype(int)

lgb_test_amount = (
    lgb_regressor
    .predict(X_test)
)

lgb_test_amount = np.maximum(
    lgb_test_amount,
    0
)

lgb_final_prediction = np.where(
    lgb_test_rain_prediction == 1,
    lgb_test_amount,
    0
)

Classification comparison

In [ ]:
classification_results = []

classification_models = {
    "Random Forest": (
        rf_test_rain_prediction,
        rf_test_probability
    ),

    "XGBoost": (
        xgb_test_rain_prediction,
        xgb_test_probability
    ),

    "LightGBM": (
        lgb_test_rain_prediction,
        lgb_test_probability
    )
}

for model_name, (
    prediction,
    probability
) in classification_models.items():

    precision = precision_score(
        y_test_class,
        prediction,
        zero_division=0
    )

    recall = recall_score(
        y_test_class,
        prediction,
        zero_division=0
    )

    f1 = f1_score(
        y_test_class,
        prediction,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test_class,
        probability
    )

    classification_results.append({
        "Model": model_name,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": roc_auc
    })

classification_results = pd.DataFrame(
    classification_results
)

classification_results

,Model,Precision,Recall,F1,ROC-AUC
0,Random Forest,0.788537,0.823067,0.805432,0.900692
1,XGBoost,0.672318,0.908585,0.772797,0.889705
2,LightGBM,0.763569,0.789703,0.776416,0.882325


Rainfall amount comparison

In [ ]:
rainfall_predictions = {

    "Random Forest":
        rf_final_prediction,

    "XGBoost":
        xgb_final_prediction,

    "LightGBM":
        lgb_final_prediction
}

In [ ]:
regression_results = []

for model_name, prediction in rainfall_predictions.items():

    mae = mean_absolute_error(
        y_test_rain,
        prediction
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test_rain,
            prediction
        )
    )

    regression_results.append({
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse
    })

regression_results = pd.DataFrame(
    regression_results
)

regression_results

,Model,MAE,RMSE
0,Random Forest,0.292491,0.763620
1,XGBoost,0.292941,0.714152
2,LightGBM,0.269936,0.706461


Heavy rainfall evaluation

In [ ]:
HEAVY_RAIN_THRESHOLD = 10.0

actual_heavy = (
    y_test_rain >= HEAVY_RAIN_THRESHOLD
).astype(int)

In [ ]:
heavy_results = []

for model_name, prediction in rainfall_predictions.items():

    predicted_heavy = (
        prediction >= HEAVY_RAIN_THRESHOLD
    ).astype(int)

    precision = precision_score(
        actual_heavy,
        predicted_heavy,
        zero_division=0
    )

    recall = recall_score(
        actual_heavy,
        predicted_heavy,
        zero_division=0
    )

    f1 = f1_score(
        actual_heavy,
        predicted_heavy,
        zero_division=0
    )

    heavy_results.append({
        "Model": model_name,
        "Heavy Rain Precision": precision,
        "Heavy Rain Recall": recall,
        "Heavy Rain F1": f1
    })

heavy_results = pd.DataFrame(
    heavy_results
)

heavy_results

,Model,Heavy Rain Precision,Heavy Rain Recall,Heavy Rain F1
0,Random Forest,0.548387,0.250000,0.343434
1,XGBoost,0.707865,0.308824,0.430034
2,LightGBM,0.623656,0.284314,0.390572


Finale Results

In [ ]:
final_results = (
    classification_results
    .merge(
        regression_results,
        on="Model"
    )
    .merge(
        heavy_results,
        on="Model"
    )
)

final_results

,Model,Precision,Recall,F1,ROC-AUC,MAE,RMSE,Heavy Rain Precision,Heavy Rain Recall,Heavy Rain F1
0,Random Forest,0.788537,0.823067,0.805432,0.900692,0.292491,0.763620,0.548387,0.250000,0.343434
1,XGBoost,0.672318,0.908585,0.772797,0.889705,0.292941,0.714152,0.707865,0.308824,0.430034
2,LightGBM,0.763569,0.789703,0.776416,0.882325,0.269936,0.706461,0.623656,0.284314,0.390572


In [ ]:
classifiers = {
    "RF": rf_classifier,
    "XGB": xgb_classifier,
    "LGBM": lgb_classifier
}

regressors = {
    "RF": rf_regressor,
    "XGB": xgb_regressor,
    "LGBM": lgb_regressor
}

In [ ]:
classifier_predictions = {}

for clf_name, clf in classifiers.items():

    print(f"Getting predictions from {clf_name} classifier...")

    class_pred = clf.predict(X_test)
    rain_prob = clf.predict_proba(X_test)[:, 1]

    classifier_predictions[clf_name] = {
        "class_pred": class_pred,
        "rain_prob": rain_prob
    }

print("All classifier predictions generated.")

Getting predictions from RF classifier...
Getting predictions from XGB classifier...
Getting predictions from LGBM classifier...
All classifier predictions generated.


In [ ]:
regressor_predictions = {}

for reg_name, reg in regressors.items():

    print(f"Getting predictions from {reg_name} regressor...")

    amount_pred = reg.predict(X_test)

    # Rainfall cannot be negative
    amount_pred = np.maximum(amount_pred, 0)

    regressor_predictions[reg_name] = amount_pred

print("All regressor predictions generated.")

Getting predictions from RF regressor...


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=2)]: Done   4 tasks      | elapsed:    0.1s
[Parallel(n_jobs=2)]: Done   9 tasks      | elapsed:    0.2s
[Parallel(n_jobs=2)]: Done  14 tasks      | elapsed:    0.2s
[Parallel(n_jobs=2)]: Done  21 tasks      | elapsed:    0.3s
[Parallel(n_jobs=2)]: Done  28 tasks      | elapsed:    0.5s
[Parallel(n_jobs=2)]: Done  37 tasks      | elapsed:    0.6s
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.7s
[Parallel(n_jobs=2)]: Done  57 tasks      | elapsed:    0.9s
[Parallel(n_jobs=2)]: Done  68 tasks      | elapsed:    1.1s
[Parallel(n_jobs=2)]: Done  81 tasks      | elapsed:    1.3s
[Parallel(n_jobs=2)]: Done  94 tasks      | elapsed:    1.5s
[Parallel(n_jobs=2)]: Done 109 tasks      | elapsed:    1.7s
[Parallel(n_jobs=2)]: Done 124 tasks      | elapsed:    2.0s
[Parallel(n_jobs=2)]: Done 141 tasks      | elapsed:    2.2s
[Para

Getting predictions from XGB regressor...
Getting predictions from LGBM regressor...
All regressor predictions generated.


Evaluate all 9 combinations

In [ ]:
# Actual rainfall amount for the 2025 test set
y_test_amount = test_tree["rainfall_mm"].values

print("y_test_amount shape:", y_test_amount.shape)
print("First 10 actual rainfall values:")
print(y_test_amount[:10])

y_test_amount shape: (96096,)
First 10 actual rainfall values:
[0.3 0.1 0.  0.  0.  0.  0.  0.  0.6 0.2]


In [ ]:
HEAVY_RAIN_THRESHOLD = 10.0

results = []

for clf_name in classifiers.keys():

    # -----------------------------
    # Classifier prediction
    # -----------------------------
    class_pred = classifier_predictions[clf_name]["class_pred"]
    rain_prob = classifier_predictions[clf_name]["rain_prob"]

    # Classification metrics
    occurrence_f1 = f1_score(
        y_test_class,
        class_pred,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test_class,
        rain_prob
    )

    for reg_name in regressors.keys():

        # -----------------------------
        # Regressor prediction
        # -----------------------------
        amount_pred = regressor_predictions[reg_name]

        # -----------------------------
        # Two-stage final prediction
        # -----------------------------
        final_pred = np.where(
            class_pred == 1,
            amount_pred,
            0
        )

        # Make sure predictions are non-negative
        final_pred = np.maximum(final_pred, 0)

        # -----------------------------
        # Overall rainfall metrics
        # -----------------------------
        mae = mean_absolute_error(
            y_test_amount,
            final_pred
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_test_amount,
                final_pred
            )
        )

        r2 = r2_score(
            y_test_amount,
            final_pred
        )

        # -----------------------------
        # Heavy rain evaluation
        # -----------------------------
        actual_heavy = (
            y_test_amount >= HEAVY_RAIN_THRESHOLD
        ).astype(int)

        predicted_heavy = (
            final_pred >= HEAVY_RAIN_THRESHOLD
        ).astype(int)

        heavy_rain_f1 = f1_score(
            actual_heavy,
            predicted_heavy,
            zero_division=0
        )

        # -----------------------------
        # Store results
        # -----------------------------
        results.append({
            "Combination": f"{clf_name} + {reg_name}",
            "Occurrence F1": occurrence_f1,
            "ROC-AUC": roc_auc,
            "MAE": mae,
            "RMSE": rmse,
            "R²": r2,
            "Heavy Rain F1": heavy_rain_f1
        })

# Convert results to DataFrame
results_df = pd.DataFrame(results)

# Round values for easier reading
results_df = results_df.round(4)

results_df

,Combination,Occurrence F1,ROC-AUC,MAE,RMSE,R²,Heavy Rain F1
0,RF + RF,0.8054,0.9007,0.2925,0.7636,0.4754,0.3434
1,RF + XGB,0.8054,0.9007,0.2710,0.7140,0.5413,0.4300
2,RF + LGBM,0.8054,0.9007,0.2676,0.7024,0.5561,0.3906
3,XGB + RF,0.7728,0.8897,0.3228,0.7688,0.4683,0.3434
4,XGB + XGB,0.7728,0.8897,0.2929,0.7142,0.5412,0.4300
5,XGB + LGBM,0.7728,0.8897,0.2892,0.7017,0.5570,0.3906
6,LGBM + RF,0.7764,0.8823,0.2945,0.7645,0.4742,0.3469
7,LGBM + XGB,0.7764,0.8823,0.2733,0.7179,0.5363,0.4300
8,LGBM + LGBM,0.7764,0.8823,0.2699,0.7065,0.5510,0.3906


In [ ]:
results_df.round(4)

,Combination,Occurrence F1,ROC-AUC,MAE,RMSE,Heavy Rain F1
0,RF + RF,0.8054,0.9007,0.2925,0.7636,0.3434
1,RF + XGB,0.8054,0.9007,0.2710,0.7140,0.4300
2,RF + LGBM,0.8054,0.9007,0.2676,0.7024,0.3906
3,XGB + RF,0.7728,0.8897,0.3228,0.7688,0.3434
4,XGB + XGB,0.7728,0.8897,0.2929,0.7142,0.4300
5,XGB + LGBM,0.7728,0.8897,0.2892,0.7017,0.3906
6,LGBM + RF,0.7764,0.8823,0.2945,0.7645,0.3469
7,LGBM + XGB,0.7764,0.8823,0.2733,0.7179,0.4300
8,LGBM + LGBM,0.7764,0.8823,0.2699,0.7065,0.3906


For heavy rain

In [ ]:
HEAVY_RAIN_THRESHOLD = 10.0

In [ ]:
actual_heavy = (
    y_test_amount >= 10.0
).astype(int)

predicted_heavy = (
    final_pred >= 10.0
).astype(int)

In [ ]:
heavy_rain_f1 = f1_score(
    actual_heavy,
    predicted_heavy,
    zero_division=0
)

In [ ]:
RESULT_DIR = "/content/drive/MyDrive/Capstone_Datasets/Tree_Results"

import os
os.makedirs(RESULT_DIR, exist_ok=True)

results_path = os.path.join(
    RESULT_DIR,
    "all_9_two_stage_rainfall_combinations_2025.csv"
)

results_df.to_csv(
    results_path,
    index=False
)

print("Saved:")
print(results_path)

Saved:
/content/drive/MyDrive/Capstone_Datasets/Tree_Results/all_9_two_stage_rainfall_combinations_2025.csv


In [ ]:
MODEL_DIR = "/content/drive/MyDrive/Capstone_Datasets/Final_Rainfall_Models"
PRED_DIR = "/content/drive/MyDrive/Capstone_Datasets/Final_Rainfall_Predictions"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PRED_DIR, exist_ok=True)


In [ ]:
import joblib

joblib.dump(
    rf_classifier,
    os.path.join(
        MODEL_DIR,
        "RF_classifier_LGBM_regressor.joblib"
    )
)

joblib.dump(
    lgb_regressor,
    os.path.join(
        MODEL_DIR,
        "LGBM_regressor_for_RF_classifier.joblib"
    )
)


['/content/drive/MyDrive/Capstone_Datasets/Final_Rainfall_Models/LGBM_regressor_for_RF_classifier.joblib']

In [ ]:
# RF classifier prediction
rf_class_pred = rf_classifier.predict(X_test)

rf_rain_probability = rf_classifier.predict_proba(X_test)[:, 1]

# LightGBM rainfall amount prediction
lgbm_amount_pred = lgb_regressor.predict(X_test)

# Remove negative rainfall predictions
lgbm_amount_pred = np.maximum(lgbm_amount_pred, 0)

# Complete two-stage prediction
rf_lgbm_final_pred = np.where(
    rf_class_pred == 1,
    lgbm_amount_pred,
    0
)

rf_lgbm_final_pred = np.maximum(
    rf_lgbm_final_pred,
    0
)

In [ ]:
rf_lgbm_predictions = test_tree[
    [
        "district",
        "pointid",
        "latitude",
        "longitude",
        "datetime",
        "rainfall_mm",
        "rain_occurs"
    ]
].copy()

rf_lgbm_predictions["predicted_rain_occurs"] = rf_class_pred
rf_lgbm_predictions["rain_probability"] = rf_rain_probability
rf_lgbm_predictions["predicted_rainfall_mm"] = rf_lgbm_final_pred

rf_lgbm_path = os.path.join(
    PRED_DIR,
    "RF_LGBM_final_rainfall_predictions_2025.csv"
)

rf_lgbm_predictions.to_csv(
    rf_lgbm_path,
    index=False
)

print("RF + LGBM predictions saved:")
print(rf_lgbm_path)

RF + LGBM predictions saved:
/content/drive/MyDrive/Capstone_Datasets/Final_Rainfall_Predictions/RF_LGBM_final_rainfall_predictions_2025.csv
